In [ ]:
# Installs the required libraries. This takes about one minute.
!pip install -q pypdf sentence-transformers "gradio>=6,<7" google-genai

In [ ]:
import subprocess
import sys

def check_system_dependencies():
    """Verify and install system-level packages needed for high-quality OCR and PDF rendering."""
    print("=== System Dependency Diagnosis ===")

    # Check Tesseract
    try:
        tesseract_ver = subprocess.check_output(["tesseract", "--version"], stderr=subprocess.STDOUT).decode()
        print(f"[✓] Tesseract is available:\n{tesseract_ver.splitlines()[0]}")
    except Exception:
        print("[!] Tesseract OCR not found. Installing tesseract-ocr...")
        subprocess.run(["apt-get", "update", "-y"], stdout=subprocess.DEVNULL)
        subprocess.run(["apt-get", "install", "-y", "tesseract-ocr"], stdout=subprocess.DEVNULL)
        print("[✓] Tesseract OCR installed successfully.")

check_system_dependencies()

=== System Dependency Diagnosis ===
[✓] Tesseract is available:
tesseract 5.3.4


In [ ]:
import os
import numpy as np
from sentence_transformers import SentenceTransformer

# Initialize the sentence transformer embedder globally once to prevent reload leaks
if 'embedder' not in globals():
    print("Initializing SentenceTransformer embedder...")
    embedder = SentenceTransformer(EMBED_MODEL_NAME)
    print("Embedder loaded successfully.")
else:
    print("Using already initialized embedder.")

Using already initialized embedder.


In [ ]:
# STUDENT TODO: the Gemma 4 model that writes the answers.
#   "gemma-4-26b-a4b-it"  Mixture of Experts, 26B total / about 4B active parameters. Fast. (default)
#   "gemma-4-31b-it"      Dense, 31B parameters. More capable, slower.
GEMMA_MODEL = "gemma-4-26b-a4b-it"

# STUDENT TODO: the number of words in each chunk.
#   Small chunks (50)   more precise matches, but may lose context
#   Large chunks (300)  more context, but less precise and longer prompts
CHUNK_SIZE = 120

# STUDENT TODO: the number of words shared by neighbouring chunks, so sentences are not split.
CHUNK_OVERLAP = 30

# STUDENT TODO: the number of chunks given to Gemma for each question.
TOP_K = 3

# STUDENT TODO: the rules Gemma follows when answering. The rule to use only the context prevents invented answers.
ANSWER_RULES = """You are an Open-Source Contributor Assistant.
Answer questions using ONLY the retrieved context passages.
Each passage identifies its source filename and page.
Cite every factual answer using the source filename and page,
for example (README.md, page 1).
If the documents do not support an answer, reply exactly:
"The answer could not be found in the provided documents."
Do not invent commands, setup steps, or contribution rules.
Keep answers concise and useful for first-time contributors."""


TEMPERATURE = 0.2       # a low value keeps answers consistent and close to the document

EMBED_MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"  # STUDENT TODO (optional): try "BAAI/bge-small-en-v1.5"
print("Settings configured.")

Settings configured.


In [ ]:


# STUDENT TODO: replace with text relevant to your problem statement, or upload a PDF in the application instead.
SAMPLE_PAGES = [
    """CS 101: Introduction to Programming (Sample Syllabus). Instructor: Dr. A. Rao. Email: arao@example.edu.
    Lectures are held Monday and Wednesday, 10:00 to 11:30 AM, in Room B-204. Labs are on Friday from 2:00 to 4:00 PM
    in the Computer Lab (Room C-110). Office hours: Tuesday 3:00 to 5:00 PM, or by appointment. This course introduces
    programming using Python. No prior coding experience is required.""",

    """Grading. Weekly lab exercises: 20%. Two mini-projects: 30% (15% each). Midterm exam: 20%. Final exam: 25%.
    Class participation: 5%. A score of 90% or above earns an A, 80 to 89% a B, 70 to 79% a C, and 60 to 69% a D.
    Late submissions lose 10% per day, up to a maximum of 3 days, after which they receive zero. Each student has
    two grace days per semester that may be used to submit any lab late without penalty.""",

    """Schedule. Weeks 1-2: variables, data types, and input/output. Weeks 3-4: conditions and loops. Week 5: functions.
    Week 6: lists and dictionaries. Week 7: Midterm exam (in class, closed book). Weeks 8-9: files and error handling.
    Weeks 10-11: an introduction to APIs and working with data. Weeks 12-13: mini-project 2, a team project of up to
    3 students. Week 14: review. The final exam is a 2-hour written exam during the examination period.""",

    """Policies. Attendance: students must attend at least 75% of lectures to be eligible for the final exam.
    Academic integrity: students may discuss ideas with classmates, but all submitted code must be their own. AI
    tools may be used for learning and debugging, but their use must be stated in a comment at the top of the file.
    Laptops: bring a laptop to every lab. Students without a laptop may reserve a machine in the Computer Lab.
    Textbook: "Automate the Boring Stuff with Python" (available free online).""",
]
print(f"Sample document contains {len(SAMPLE_PAGES)} pages.")

Sample document contains 4 pages.


In [ ]:
import os
import numpy as np
from pypdf import PdfReader

# One shared knowledge base for all uploaded documents.
knowledge_base = {
    "name": None,
    "chunks": [],
    "vectors": None,
    "files": []
}

# We will use the enhanced read_document function defined in pwXTvJq5BDUt
# which includes OCR preprocessing, scanned PDF rendering, and support for JPG/PNG.

def split_into_chunks(pages, filename):
    """Split text into overlapping chunks with source information."""
    chunks = []
    step = max(1, CHUNK_SIZE - CHUNK_OVERLAP)

    for page_number, page_text in enumerate(pages, start=1):
        words = page_text.split()

        for start in range(0, len(words), step):
            text = " ".join(words[start:start + CHUNK_SIZE])

            if len(text.split()) > 5:
                chunks.append({
                    "filename": filename,
                    "page": page_number,
                    "text": text
                })

            if start + CHUNK_SIZE >= len(words):
                break

    return chunks


def build_knowledge_base(pages, name, replace=False):
    """Embed a document and add it to the shared knowledge base."""
    new_chunks = split_into_chunks(pages, name)

    if not new_chunks:
        raise ValueError(f"No readable text found in {name}.")

    if replace:
        all_chunks = new_chunks
        all_files = [name]
    else:
        all_chunks = knowledge_base["chunks"] + new_chunks
        all_files = list(knowledge_base["files"])

        if name not in all_files:
            all_files.append(name)

    # Create embeddings for all current chunks.
    vectors = embedder.encode(
        [chunk["text"] for chunk in all_chunks],
        normalize_embeddings=True
    )

    # Update the knowledge base only after embedding succeeds.
    knowledge_base["chunks"] = all_chunks
    knowledge_base["vectors"] = vectors
    knowledge_base["files"] = all_files
    knowledge_base["name"] = ", ".join(all_files)

    return (
        f"Loaded {name}: {len(pages)} pages, "
        f"{len(new_chunks)} chunks. "
        f"Total documents: {len(all_files)}; "
        f"total chunks: {len(all_chunks)}."
    )


def retrieve(question, top_k=TOP_K):
    """Find relevant chunks across every uploaded document."""
    if not knowledge_base["chunks"]:
        raise ValueError("Upload a document before asking questions.")

    question_vector = embedder.encode(
        [question], normalize_embeddings=True
    )[0]

    similarities = knowledge_base["vectors"] @ question_vector
    best = np.argsort(similarities)[::-1][:top_k]

    return [
        {
            **knowledge_base["chunks"][i],
            "score": float(similarities[i])
        }
        for i in best
    ]


def answer(question):
    """Answer using retrieved passages and source-aware citations."""
    found = retrieve(question)

    context = "\n\n".join(
        f"[Source: {c['filename']}, page {c['page']}] {c['text']}"
        for c in found
    )

    prompt = (
        "Use the following retrieved passages to answer the question.\n"
        "Cite the source filename and page for factual claims.\n"
        "If the answer is not supported by the passages, say so.\n\n"
        f"Context:\n{context}\n\nQuestion: {question}"
    )

    demo = (
        "Demonstration mode: retrieval worked, but Gemma did not generate "
        "an answer because no API key is active.\n\n"
        + "\n\n".join(
            f"Source: {c['filename']}, page {c['page']}\n{c['text'][:400]}"
            for c in found
        )
    )

    reply = generate(
        prompt,
        system_text=ANSWER_RULES,
        demo_reply=demo
    )

    return reply, found

In [ ]:
import os
import pytesseract
import pypdfium2 as pdfium
from pypdf import PdfReader
from PIL import Image, ImageOps, ImageEnhance, ImageFilter

def preprocess_image_variants(image):
    """Generate multiple preprocessed variations of an image to maximize OCR readability."""
    variants = []

    # Variant 1: Original in Grayscale
    gray = ImageOps.grayscale(image)
    variants.append(gray)

    # Variant 2: High Contrast Grayscale
    enhancer = ImageEnhance.Contrast(gray)
    high_contrast = enhancer.enhance(2.5)
    variants.append(high_contrast)

    # Variant 3: Sharpened + High Contrast + Scale-up
    scaled = high_contrast.resize((high_contrast.width * 2, high_contrast.height * 2), Image.Resampling.LANCZOS)
    sharpened = scaled.filter(ImageFilter.SHARPEN)
    variants.append(sharpened)

    # Variant 4: Binary Thresholding (for light/shadowed documents)
    threshold_val = 127
    binary = gray.point(lambda p: 255 if p > threshold_val else 0)
    variants.append(binary)

    return variants

def read_document(path):
    """Read PDF (including scanned), Markdown/text, and common image formats (JPG, PNG) using robust fallback OCR."""
    ext = os.path.splitext(path)[1].lower()

    # Handle Markdown and Text Files
    if ext in [".md", ".txt"]:
        with open(path, "r", encoding="utf-8", errors="replace") as f:
            return [f.read()]

    # Handle PDF Files (including OCR fallback for scanned pages)
    if ext == ".pdf":
        reader = PdfReader(path)
        pages = [(page.extract_text() or "") for page in reader.pages]

        pdf = None
        for i, text in enumerate(pages):
            if len(text.strip()) < 40:
                if pdf is None:
                    pdf = pdfium.PdfDocument(path)

                page = pdf[i]
                bitmap = page.render(scale=3)  # High resolution render
                image = bitmap.to_pil()

                # Try image variants for scanned PDF page
                extracted_text = ""
                for variant in preprocess_image_variants(image):
                    txt = pytesseract.image_to_string(variant, lang="eng").strip()
                    if len(txt) > len(extracted_text):
                        extracted_text = txt

                pages[i] = extracted_text if extracted_text else text
                page.close()

        if pdf is not None:
            pdf.close()

        if not any(text.strip() for text in pages):
            raise ValueError(
                "No text could be extracted. Check the PDF quality "
                "or try a clearer scan."
            )

        return pages

    # Handle Image Files (Aadhar, PAN, photos, etc.)
    if ext in [".jpg", ".jpeg", ".png"]:
        try:
            image = Image.open(path)
            extracted_text = ""

            # Try multiple pre-processing variations until we successfully extract readable text
            for variant in preprocess_image_variants(image):
                txt = pytesseract.image_to_string(variant, lang="eng").strip()
                if len(txt) > len(extracted_text):
                    extracted_text = txt

            if not extracted_text.strip():
                # Fallback to standard pytesseract run on original
                extracted_text = pytesseract.image_to_string(image, lang="eng").strip()

            if not extracted_text.strip():
                raise ValueError("OCR returned no readable text for this image. Please ensure the image has clear visible text.")

            return [extracted_text]
        except Exception as e:
            raise ValueError(f"Failed to process image file with OCR: {e}")

    raise ValueError("Unsupported file type. Use PDF, MD, TXT, JPG, JPEG, or PNG.")

In [ ]:
def generate_onboarding_guide():
    """
    Runs the onboarding workflow (Feature E).
    Queries the loaded documents for critical beginner information and generates a structured checklist.
    """
    if not knowledge_base["chunks"]:
        raise ValueError("Please upload project documents first.")

    # 1. Search for key onboarding keywords to retrieve the most relevant passages
    onboarding_query = "installation setup prerequisites run tests contribution guidelines pull request"
    found = retrieve(onboarding_query, top_k=min(5, len(knowledge_base["chunks"])))

    context = "\n\n".join(
        f"[Source: {c['filename']}, page {c['page']}] {c['text']}"
        for c in found
    )

    # 2. Instruct Gemma with specific requirements for the onboarding workflow
    system_prompt = """You are an Open-Source Contributor Assistant. Your task is to generate a beginner-friendly onboarding checklist.

Structure the checklist under these 5 categories:
1. Project Prerequisites
2. Installation and Setup Steps
3. How to Run or Test the Project
4. Contribution Guidelines and Code Standards
5. How to Report Issues and Submit a Pull Request

For each category:
- If the retrieved context contains explicit information for that category, write a bulleted list of setup items or guidelines. Every single point must include a citation of the source filename and page (e.g. [README.md, page 1]).
- If the retrieved context does not contain any relevant information for a category, output exactly 'Information not found in documents.' under that category.

Never invent commands, prerequisites, or contribution steps that are not explicitly detailed in the provided context."""

    prompt = (
        "Based on the retrieved context below, construct the beginner onboarding checklist.\n\n"
        f"Context:\n{context}\n\n"
        "Generate Onboarding Checklist:"
    )

    demo_reply = (
        "### Beginner Onboarding Checklist (Demonstration Mode)\n\n"
        "1. **Project Prerequisites**\n"
        "   - Information not found in documents.\n\n"
        "2. **Installation and Setup Steps**\n"
        "   - Laptops: Bring a laptop to every lab. If you do not have one, you can reserve a machine in the Computer Lab (Sample CS 101 syllabus, page 4)\n\n"
        "3. **How to Run or Test the Project**\n"
        "   - Lectures are Monday/Wednesday in Room B-204 and Labs are Friday 2:00-4:00 PM in Room C-110 (Sample CS 101 syllabus, page 1)\n\n"
        "4. **Contribution Guidelines and Code Standards**\n"
        "   - All code must be your own. AI tools are allowed for learning and debugging, but must be declared in a comment at the top of the file (Sample CS 101 syllabus, page 4)\n\n"
        "5. **How to Report Issues and Submit a Pull Request**\n"
        "   - Submit lab assignments using the grace days system: two grace days per semester (Sample CS 101 syllabus, page 2)"
    )

    reply = generate(
        prompt,
        system_text=system_prompt,
        demo_reply=demo_reply
    )

    return reply, found

In [ ]:
import json
import pandas as pd
import time

# User-specific persistent session state for checklists
user_checklist_state = {
    "1. Project Prerequisites": False,
    "2. Installation and Setup Steps": False,
    "3. How to Run or Test the Project": False,
    "4. Contribution Guidelines and Code Standards": False,
    "5. How to Report Issues and Submit a Pull Request": False
}

# Session chat memory
chat_history = []  # List of tuples (user_message, assistant_response)

def reset_session():
    """Reset active conversation history and user checklist statuses."""
    global chat_history, user_checklist_state
    chat_history = []
    for key in user_checklist_state:
        user_checklist_state[key] = False
    return "Session history and checklist states cleared successfully!"

def generate_readiness_report():
    """Feature 2: Repository Readiness Report."""
    if not knowledge_base["chunks"]:
        return "Please upload project documentation files to analyze readiness."

    query = "project purpose installation testing contribution pull request code of conduct"
    found = retrieve(query, top_k=min(6, len(knowledge_base["chunks"])))
    context = "\n\n".join(f"[Source: {c['filename']}, page {c['page']}] {c['text']}" for c in found)

    system_prompt = """You are a Repository Quality Auditor.
Analyze the retrieved documentation context and generate a Markdown Repository Readiness Report.
Structure the report with the following headers:
- ## Executive Summary
- ## Project Purpose & Vision
- ## Setup & Installation Health (Mark as 'Missing' if not found)
- ## Testing Environment & Instructions (Mark as 'Missing' if not found)
- ## Contribution Process & Pull Request Rules
- ## Documentation Gaps & Recommended Actions

For each category, provide specific evidence citations or state 'not found in uploaded documents'. Do not invent details."""

    prompt = f"Retrieved context:\n{context}\n\nGenerate detailed Repository Readiness Report:"
    demo_reply = "## Executive Summary\nThis report evaluates the repository quality of the uploaded documentation. Overall status: Partial Readiness.\n\n## Setup & Installation Health\n- Found instructions regarding laptop setups on Page 4 of CS 101 syllabus. Detailed code build instructions are missing."

    report = generate(prompt, system_text=system_prompt, demo_reply=demo_reply)
    return report

def analyze_documentation_gaps():
    """Feature 7: Documentation Gap Analyzer."""
    if not knowledge_base["chunks"]:
        return "No documents uploaded yet to evaluate gaps."

    test_queries = [
        "How do I configure the external configuration variables?",
        "What command runs unit tests?",
        "Where is the linting or style guide specification?",
        "How do I configure a local development environment database?"
    ]

    gaps = []
    for q in test_queries:
        results = retrieve(q, top_k=2)
        best_score = results[0]["score"] if results else 0.0
        if best_score < 0.35:
            gaps.append(f"- **Gap Identified for '{q}'**: Maximum context relevance score was very low ({best_score:.2f}). No explicit local setup steps or configurations were found to answer this developer query.")

    if not gaps:
        return "Awesome! The uploaded documentation is highly comprehensive and covers local setup, styling guidelines, and automated tests."

    return "### 🔍 Documentation Gap Analysis\nBased on semantic search evaluations on typical contributor questions, the following documentation gaps were identified:\n\n" + "\n".join(gaps) + "\n\n*Recommendation: Please supplement your documentation with setup code blocks or instructions describing these parameters.*"

In [ ]:
# Evaluation Suite Dataset (Feature 9)
EVALUATION_DATASET = [
    {"question": "What is the email address of the instructor?", "expected": "arao@example.edu", "refusal_expected": False},
    {"question": "Where are lectures held?", "expected": "Room B-204", "refusal_expected": False},
    {"question": "When are labs held?", "expected": "Friday from 2:00 to 4:00 PM", "refusal_expected": False},
    {"question": "What percentage of the grade are weekly lab exercises?", "expected": "20%", "refusal_expected": False},
    {"question": "What is the policy for late submissions?", "expected": "lose 10% per day, up to a maximum of 3 days", "refusal_expected": False},
    {"question": "What percentage of lectures must students attend to be eligible for the final exam?", "expected": "at least 75%", "refusal_expected": False},
    {"question": "What is the textbook of the course?", "expected": "Automate the Boring Stuff with Python", "refusal_expected": False},
    {"question": "What are the office hours?", "expected": "Tuesday 3:00 to 5:00 PM", "refusal_expected": False},
    {"question": "How many grace days does a student have per semester?", "expected": "two grace days", "refusal_expected": False},
    {"question": "What is the schedule for Weeks 1-2?", "expected": "variables, data types, and input/output", "refusal_expected": False},
    {"question": "How do I configure the production PostgreSQL database credentials?", "expected": "The answer could not be found in the provided documents.", "refusal_expected": True},
    {"question": "What command do I run to compile the C++ bindings?", "expected": "The answer could not be found in the provided documents.", "refusal_expected": True},
    {"question": "Is there any rule about class participation grade percentage?", "expected": "5%", "refusal_expected": False},
    {"question": "Where are the office hours held?", "expected": "The answer could not be found in the provided documents.", "refusal_expected": True},
    {"question": "How many mini-projects are there?", "expected": "Two mini-projects", "refusal_expected": False}
]

def run_evaluation_suite():
    """Executes 15 test evaluations to track RAG metrics."""
    if not knowledge_base["chunks"]:
        return "Please load the CS 101 sample dataset first before running the benchmark suite."

    results = []
    correct_count = 0
    total_latency = 0.0

    for i, item in enumerate(EVALUATION_DATASET):
        start_time = time.time()
        try:
            ans, passages = answer(item["question"])
            latency = time.time() - start_time
            total_latency += latency

            # Simple exact-match keyword heuristic or refusal matching for benchmark scores
            passed = False
            if item["refusal_expected"]:
                if "could not be found" in ans.lower() or "unable to find" in ans.lower():
                    passed = True
            else:
                keywords = item["expected"].split()
                matched_kws = sum(1 for kw in keywords if kw.lower() in ans.lower())
                if matched_kws / len(keywords) >= 0.5:
                    passed = True

            if passed:
                correct_count += 1

            results.append({
                "ID": i + 1,
                "Question": item["question"],
                "Passed Test": "✓ PASS" if passed else "✗ FAIL",
                "Latency (s)": f"{latency:.2f}s",
                "Response": ans[:120] + "..."
            })
        except Exception as e:
            results.append({
                "ID": i + 1,
                "Question": item["question"],
                "Passed Test": f"✗ ERROR ({str(e)})",
                "Latency (s)": "0.0s",
                "Response": "None"
            })

    score_pct = (correct_count / len(EVALUATION_DATASET)) * 100
    avg_latency = total_latency / len(EVALUATION_DATASET)

    summary_text = f"### 📊 Hackathon RAG Benchmark Evaluation Suite\n"
    summary_text += f"*   **Overall Accuracy/Match Rate**: `{score_pct:.1f}%` ({correct_count}/15 test queries)\n"
    summary_text += f"*   **Average Query Latency**: `{avg_latency:.2f}s`\n\n"

    df = pd.DataFrame(results)
    return summary_text + df.to_markdown(index=False)

In [ ]:
import json

def extract_document_metadata(text, filename):
    """
    Hackathon Feature: Uses Gemma to automatically parse, clean, and extract structured
    entities (Names, IDs, Document Type, Dates) from the raw document content.
    """
    system_prompt = """You are an expert Document Information Extractor.
Analyze the text extracted from the document and return a valid JSON object with the following fields:
- "document_type": (e.g., Admission Order, Syllabus, Aadhaar Card, PAN Card, Unknown)
- "primary_name": (The name of the student, cardholder, or instructor if found; otherwise null)
- "identifiers": (Any ID, roll number, registration number, or PAN/Aadhaar number; otherwise null)
- "important_dates": (A list of deadlines, dates of issue, or class schedules if mentioned; otherwise empty list)
- "summary": (A concise one-sentence description of the document)

Respond ONLY with a valid JSON block starting with { and ending with }. Do not include any explanations."""

    prompt = f"Extract information from this document text:\n\n{text[:4000]}\n\nJSON Output:"

    demo_reply = json.dumps({
        "document_type": "Scanned Document / ID",
        "primary_name": "Extracted Name",
        "identifiers": "Not found",
        "important_dates": [],
        "summary": "Document containing structured registration details."
    }, indent=2)

    try:
        raw_reply = generate(prompt, system_text=system_prompt, demo_reply=demo_reply)
        # Clean up any markdown wrappers like ```json if present
        clean_reply = raw_reply.strip()
        if clean_reply.startswith("```"):
            lines = clean_reply.splitlines()
            if lines[0].startswith("```"):
                lines = lines[1:]
            if lines[-1].startswith("```"):
                lines = lines[:-1]
            clean_reply = "\n".join(lines).strip()

        return json.loads(clean_reply)
    except Exception as e:
        return {
            "document_type": "Unknown",
            "primary_name": f"Error parsing: {str(e)}",
            "identifiers": None,
            "important_dates": [],
            "summary": "Could not automatically structure metadata."
        }

In [ ]:
import gradio as gr
import os

# Ensure all existing ports are closed cleanly
try:
    gr.close_all()
except Exception:
    pass

APP_TITLE = "⚡ Contributor Assistant Pro"
APP_DESCRIPTION = (
    "An ultra-responsive, streamlined AI workspace. Ingest PDFs (including scanned documents), "
    "markdown, or text files to instantly extract facts, discover guidelines, and ask questions."
)

def on_upload_simple(files):
    if not files:
        return "⚠️ Waiting for document uploads."
    messages = []
    for i, path in enumerate(files):
        filename = os.path.basename(path)
        try:
            pages = read_document(path)
            message = build_knowledge_base(pages, filename, replace=(i == 0))
            messages.append(f"✓ {message}")
        except Exception as e:
            messages.append(f"✗ Error loading {filename}: {str(e)}")
    return "\n".join(messages)

def load_sample_simple():
    try:
        status = build_knowledge_base(SAMPLE_PAGES, "Sample CS 101 syllabus", replace=True)
        return f"✓ {status}"
    except Exception as e:
        return f"✗ Error loading sample: {str(e)}"

def on_ask(question):
    if not question.strip():
        return "⚠️ Please enter a valid question.", "No question provided."
    try:
        reply, citations = answer(question)
        citation_details = []
        for idx, item in enumerate(citations, 1):
            citation_details.append(
                f"**[{idx}] Source:** {item['filename']} (Page {item['page']})\n"
                f"*Similarity Score:* {item['score']:.4f}\n"
                f"```text\n{item['text']}\n```\n"
            )
        citations_str = "\n\n".join(citation_details) if citation_details else "No sources found."
        return reply, citations_str
    except Exception as e:
        return f"❌ Error analyzing query: {str(e)}", ""

# Premium customized theme colors
custom_theme = gr.themes.Default(
    primary_hue="indigo",
    secondary_hue="violet",
    neutral_hue="slate"
).set(
    body_background_fill="#090d16",
    body_background_fill_dark="#090d16",
    button_primary_background_fill="#4f46e5",
    button_primary_text_color="#ffffff",
    block_background_fill="#111827",
    block_border_color="#1f2937",
    block_title_text_color="#f3f4f6",
    body_text_color="#e5e7eb"
)

with gr.Blocks(title=APP_TITLE) as unified_demo:
    gr.HTML("""
    <div style='text-align: center; padding: 25px 0; border-bottom: 1px solid #1f2937; margin-bottom: 25px;'>
        <h1 style='color: #6366f1; font-size: 2.5rem; font-weight: 800; margin: 0; font-family: system-ui, sans-serif;'>⚡ Contributor Assistant Pro</h1>
        <p style='color: #9ca3af; font-size: 1.1rem; margin-top: 8px;'>Instant, Factual Document Question Answering System</p>
    </div>
    """)

    with gr.Row():
        # Left Column: Resource loading
        with gr.Column(scale=2):
            gr.Markdown("### 📂 1. Document Control Center")
            file_input = gr.File(
                label="Upload PDF, TXT, MD, PNG, or JPG files",
                file_types=[".pdf", ".md", ".txt", ".jpg", ".jpeg", ".png"],
                file_count="multiple",
                type="filepath"
            )
            with gr.Row():
                upload_btn = gr.Button("🚀 Load Documents", variant="primary")
                sample_btn = gr.Button("🧪 Use Practice Dataset", variant="secondary")

            status_md = gr.Markdown("**System Status:** Ready to ingest files. Click 'Use Practice Dataset' to run on our pre-loaded syllabus context.")

        # Right Column: Interactive QA console
        with gr.Column(scale=3):
            gr.Markdown("### 💬 2. Query Workspace")
            question_box = gr.Textbox(
                label="Enter your document-specific question:",
                placeholder="e.g., What happens if I submit my assignment late?",
                lines=2
            )
            ask_btn = gr.Button("🔍 Search & Generate Answer", variant="primary")

            gr.Markdown("#### 🎯 Answer")
            answer_md = gr.Markdown("*Your answer will appear here once context is uploaded and a question is submitted.*")

            with gr.Accordion("🔍 Citations & Sources Explorer", open=False):
                sources_md = gr.Markdown("No citations generated yet.")

    # Setup Event Bindings
    upload_btn.click(
        on_upload_simple,
        inputs=file_input,
        outputs=status_md
    )

    sample_btn.click(
        load_sample_simple,
        outputs=status_md
    )

    ask_btn.click(
        on_ask,
        inputs=question_box,
        outputs=[answer_md, sources_md]
    )

    question_box.submit(
        on_ask,
        inputs=question_box,
        outputs=[answer_md, sources_md]
    )

# Pass custom_theme inside launch for Gradio 6.0 compatibility
unified_demo.launch(share=True, theme=custom_theme)

Closing server running on port: 7860
Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://b6bc82c3ecda6f43a0.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
